<a href="https://colab.research.google.com/github/efeyucel82/skan-w10-value/blob/main/notebooks/01_synthetic_world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
import numpy as np
import pandas as pd

# Rastgele sayı üreticisi. 42 sabit kaldıkça her çalıştırmada aynı veri çıkar.
rng = np.random.default_rng(42)

# Dünyanın ölçeği
N_USERS = 100_000
HORIZON_WEEKS = 10

# Gelir varsayımları
WEEKLY_PRICE = 7.99   # haftalık abonelik, dolar
IAP_PRICE = 3.99      # ekstra ses paketi, dolar
STORE_FEE = 0.30      # Apple ve Google komisyonu

NET_WEEKLY = WEEKLY_PRICE * (1 - STORE_FEE)
NET_IAP = IAP_PRICE * (1 - STORE_FEE)

print(f"Haftalık net gelir: {NET_WEEKLY:.2f} $")
print(f"IAP net gelir: {NET_IAP:.2f} $")



Haftalık net gelir: 5.59 $
IAP net gelir: 2.79 $


In [68]:
channels = pd.DataFrame({
    "channel":       ["Snapchat", "TikTok", "Meta", "Google"],
    "cpm":           [8.0,        11.0,     14.0,   20.0],   # bin gösterimin maliyeti, $
    "ipm":           [7.0,        6.0,      5.0,    4.0],    # bin gösterimde kaç install
    "ctr":           [0.016,      0.019,    0.010,  0.007],  # tıklama oranı
    "intent_mu":     [-0.6,       -0.2,     0.2,    0.5],    # ortalama niyet
    "cliff_share":   [0.0,        0.45,     0.0,    0.0],    # heves kullanıcısı payı
}).set_index("channel")

# CPI = CPM / IPM
channels["cpi"] = channels["cpm"] / channels["ipm"]

channels

,cpm,ipm,ctr,intent_mu,cliff_share,cpi
channel,,,,,,
Snapchat,8.0,7.0,0.016,-0.6,0.00,1.142857
TikTok,11.0,6.0,0.019,-0.2,0.45,1.833333
Meta,14.0,5.0,0.010,0.2,0.00,2.800000
Google,20.0,4.0,0.007,0.5,0.00,5.000000


In [69]:
countries = pd.DataFrame({
    "country_tier": ["ABD", "Tier1", "Tier2"],
    "cpm_mult":     [1.0,   0.7,     0.3],    # reklam maliyeti çarpanı
    "intent_bonus": [0.3,   0.1,     -0.4],   # niyete eklenen
    "price_mult":   [1.0,   0.9,     0.5],    # yerel fiyat çarpanı
}).set_index("country_tier")

platforms = pd.DataFrame({
    "platform":     ["iOS", "Android"],
    "cpm_mult":     [1.2,   0.8],
    "intent_bonus": [0.15,  0.0],
}).set_index("platform")

creatives = pd.DataFrame({
    "archetype":    ["hype", "demo", "ugc"],
    "ipm_mult":     [1.4,    0.8,    1.0],    # install üretme gücü çarpanı
    "ctr_mult":     [1.5,    0.8,    1.1],    # tıklama çarpanı
    "intent_bonus": [-0.4,   0.3,    0.0],
}).set_index("archetype")

print("Hücre sayısı:", len(channels) * len(countries) * len(platforms) * len(creatives))
countries, platforms, creatives


Hücre sayısı: 72


(              cpm_mult  intent_bonus  price_mult
 country_tier                                    
 ABD                1.0           0.3         1.0
 Tier1              0.7           0.1         0.9
 Tier2              0.3          -0.4         0.5,
           cpm_mult  intent_bonus
 platform                        
 iOS            1.2          0.15
 Android        0.8          0.00,
            ipm_mult  ctr_mult  intent_bonus
 archetype                                  
 hype            1.4       1.5          -0.4
 demo            0.8       0.8           0.3
 ugc             1.0       1.1           0.0)

In [70]:
CHANNEL_BUDGET = 40_000   # her kanala ayrılan test bütçesi, $

# 72 hücreyi oluştur: her kombinasyon bir satır
cells = (channels.reset_index()[["channel"]]
         .merge(countries.reset_index()[["country_tier"]], how="cross")
         .merge(platforms.reset_index()[["platform"]], how="cross")
         .merge(creatives.reset_index()[["archetype"]], how="cross"))

# Bütçe payları
country_share = {"ABD": 0.5, "Tier1": 0.3, "Tier2": 0.2}
platform_share = {"iOS": 0.6, "Android": 0.4}
cells["country_share"] = cells["country_tier"].map(country_share)

# Simpson tuzağı: Meta'da demo bütçesi Tier 2'ye, hype bütçesi ABD'ye kaymış
meta_demo = (cells["channel"] == "Meta") & (cells["archetype"] == "demo")
meta_hype = (cells["channel"] == "Meta") & (cells["archetype"] == "hype")
cells.loc[meta_demo, "country_share"] = cells.loc[meta_demo, "country_tier"].map({"ABD": 0.15, "Tier1": 0.15, "Tier2": 0.70})
cells.loc[meta_hype, "country_share"] = cells.loc[meta_hype, "country_tier"].map({"ABD": 0.70, "Tier1": 0.20, "Tier2": 0.10})

cells["spend"] = (CHANNEL_BUDGET * cells["country_share"]
                  * cells["platform"].map(platform_share) / len(creatives))

# Küçük örneklem ve SKAN körlüğü: demo, Snapchat ve TikTok'ta yeni test ediliyor
new_test = cells["channel"].isin(["Snapchat", "TikTok"]) & (cells["archetype"] == "demo")
cells.loc[new_test, "spend"] *= 0.15

print("Toplam harcama:", round(cells["spend"].sum()), "$")
print(cells.groupby("channel")["spend"].sum().round())
cells.head(10)

Toplam harcama: 137333 $
channel
Google      40000.0
Meta        40000.0
Snapchat    28667.0
TikTok      28667.0
Name: spend, dtype: float64


,channel,country_tier,platform,archetype,country_share,spend
0,Snapchat,ABD,iOS,hype,0.5,4000.000000
1,Snapchat,ABD,iOS,demo,0.5,600.000000
2,Snapchat,ABD,iOS,ugc,0.5,4000.000000
3,Snapchat,ABD,Android,hype,0.5,2666.666667
4,Snapchat,ABD,Android,demo,0.5,400.000000
5,Snapchat,ABD,Android,ugc,0.5,2666.666667
6,Snapchat,Tier1,iOS,hype,0.3,2400.000000
7,Snapchat,Tier1,iOS,demo,0.3,360.000000
8,Snapchat,Tier1,iOS,ugc,0.3,2400.000000
9,Snapchat,Tier1,Android,hype,0.3,1600.000000


In [71]:
# Her hücrenin gerçek oranları: kanal değeri x katman çarpanları
cells["cpm"] = (cells["channel"].map(channels["cpm"])
                * cells["country_tier"].map(countries["cpm_mult"])
                * cells["platform"].map(platforms["cpm_mult"]))
cells["ctr"] = cells["channel"].map(channels["ctr"]) * cells["archetype"].map(creatives["ctr_mult"])
cells["ipm"] = cells["channel"].map(channels["ipm"]) * cells["archetype"].map(creatives["ipm_mult"])

# Harcamadan huniye
cells["impressions"] = (cells["spend"] / cells["cpm"] * 1000).round().astype(int)
cells["clicks"] = rng.binomial(cells["impressions"], cells["ctr"])
cells["installs"] = rng.poisson(cells["impressions"] * cells["ipm"] / 1000)

# Gözlenen metrikler
cells["cpi"] = cells["spend"] / cells["installs"]

print("Toplam install:", cells["installs"].sum())
print(cells.groupby("channel")[["spend", "impressions", "clicks", "installs"]].sum())
print("En küçük 5 hücre:")
print(cells.nsmallest(5, "installs")[["channel", "country_tier", "platform", "archetype", "spend", "installs"]])

Toplam install: 116173
                 spend  impressions  clicks  installs
channel                                              
Google    40000.000000      3190476   25236     13593
Meta      40000.000000      5344670   55857     26312
Snapchat  28666.666667      5716272  115425     47012
TikTok    28666.666667      4157284  100141     29256
En küçük 5 hücre:
     channel country_tier platform archetype  spend  installs
28    TikTok        Tier1  Android      demo  240.0       179
25    TikTok        Tier1      iOS      demo  360.0       196
22    TikTok          ABD  Android      demo  400.0       198
19    TikTok          ABD      iOS      demo  600.0       204
7   Snapchat        Tier1      iOS      demo  360.0       281


In [72]:
# Her install bir satır: hücreyi install sayısı kadar çoğalt
users = (cells.loc[cells.index.repeat(cells["installs"]),
                   ["channel", "country_tier", "platform", "archetype"]]
         .reset_index(names="cell_id"))
users.insert(0, "user_id", np.arange(len(users)))
n = len(users)

# Test haftası içinde kurulum günü (0 = pazartesi, 6 = pazar)
users["install_day"] = rng.integers(0, 7, n)

# Gizli niyet: hücre ortalaması + kişisel fark
users["intent_mean"] = (users["channel"].map(channels["intent_mu"])
                        + users["country_tier"].map(countries["intent_bonus"])
                        + users["platform"].map(platforms["intent_bonus"])
                        + users["archetype"].map(creatives["intent_bonus"]))
users["intent"] = users["intent_mean"] + rng.normal(0, 1.0, n)

# Takip edilebilir mi? Android'de evet, iOS'ta sadece ATT izni verenler
ATT_OPTIN_RATE = 0.15
is_ios = users["platform"] == "iOS"
users["trackable"] = ~is_ios | (rng.random(n) < ATT_OPTIN_RATE)

print("Kullanıcı sayısı:", n)
print(users.groupby("channel")["intent"].agg(["mean", "std"]).round(2))
print("iOS'ta takip edilebilir oran:", round(users.loc[is_ios, "trackable"].mean(), 3))
users.head()

Kullanıcı sayısı: 116173
          mean   std
channel             
Google    0.44  1.08
Meta      0.15  1.05
Snapchat -0.79  1.08
TikTok   -0.38  1.08
iOS'ta takip edilebilir oran: 0.152


,user_id,cell_id,channel,country_tier,platform,archetype,install_day,intent_mean,intent,trackable
0,0,0,Snapchat,ABD,iOS,hype,1,-0.55,-0.531558,False
1,1,0,Snapchat,ABD,iOS,hype,2,-0.55,-1.657559,False
2,2,0,Snapchat,ABD,iOS,hype,5,-0.55,-1.181240,False
3,3,0,Snapchat,ABD,iOS,hype,6,-0.55,-0.739693,False
4,4,0,Snapchat,ABD,iOS,hype,4,-0.55,-2.074024,False


In [73]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

z = users["intent"].to_numpy()

# Gizli tip: TikTok'tan gelen heves kullanıcıları (2. haftadan sonra toplu bırakacaklar)
users["is_tourist"] = (users["channel"].map(channels["cliff_share"]).to_numpy() > rng.random(n))
tourist = users["is_tourist"].to_numpy()

# 1. Onboarding'i bitirdi mi?
onboard = rng.random(n) < sigmoid(1.2 + 0.8 * z)

# 2. Kaç şarkı üretti? (sadece onboarding'i bitirenler)
songs = np.where(onboard, rng.poisson(np.exp(0.5 + 0.5 * z)), 0)

# 3. İlk şarkıya kadar kaç dakika geçti? (şarkı üretmeyenler için boş)
mins = np.where(songs > 0, rng.lognormal(np.log(12) - 0.4 * z, 0.8), np.nan)

# 4. Kendi sesini kaydetti mi? Heves kullanıcıları daha az kaydediyor
voice = onboard & (rng.random(n) < sigmoid(-0.8 + 0.9 * z - 1.0 * tourist))

# 5. Metinden şarkıyı denedi mi? Zayıf bir sinyal
text_song = onboard & (rng.random(n) < sigmoid(-1.0 + 0.3 * z))

# 6. Bir şarkıyı dışa aktardı ya da paylaştı mı? Heves kullanıcıları neredeyse hiç
export = (songs > 0) & (rng.random(n) < sigmoid(-1.5 + 0.8 * z + 0.15 * songs - 1.5 * tourist))

# 7. Abone oldu mu? Heves kullanıcıları ilk gün coşkuyla abone oluyor
sub = rng.random(n) < sigmoid(-4.5 + 1.7 * z + 0.6 * voice + 0.4 * export + 1.0 * tourist)

# 8. İlk hafta ekstra paket aldı mı? Sadece aboneler
iap = sub & (rng.random(n) < sigmoid(-2.2 + 0.6 * z))

users["d0_onboarding_done"] = onboard
users["d0_songs_generated"] = songs
users["d0_mins_to_first_gen"] = mins.round(1)
users["d0_voice_recorded"] = voice
users["d0_text_to_song"] = text_song
users["d0_song_exported"] = export
users["d0_subscribed"] = sub
users["d0_iap"] = iap

d0_cols = [c for c in users.columns if c.startswith("d0_")]
print(users.groupby("channel")[d0_cols].mean().round(3).T)
print("Genel abonelik oranı:", round(sub.mean(), 4))


channel               Google    Meta  Snapchat  TikTok
d0_onboarding_done     0.792   0.761     0.623   0.690
d0_songs_generated     2.048   1.695     0.918   1.209
d0_mins_to_first_gen  13.137  14.824    19.903  17.197
d0_voice_recorded      0.361   0.303     0.159   0.169
d0_text_to_song        0.244   0.224     0.154   0.180
d0_song_exported       0.277   0.228     0.100   0.104
d0_subscribed          0.105   0.071     0.019   0.048
d0_iap                 0.026   0.016     0.003   0.009
Genel abonelik oranı: 0.048


In [74]:
W = HORIZON_WEEKS
z = users["intent"].to_numpy()
voice = users["d0_voice_recorded"].to_numpy()
export = users["d0_song_exported"].to_numpy()
tourist = users["is_tourist"].to_numpy()
sub0 = users["d0_subscribed"].to_numpy()

# İlk gün abone olmayanların küçük bir kısmı ilk hafta içinde abone oluyor
late = ~sub0 & (rng.random(n) < sigmoid(-5.0 + 1.0 * z + 0.8 * export))
start_week = np.where(sub0, 1, np.where(late, 2, 0))   # 0 = hiç abone olmadı

# Her abonenin kişisel haftalık yenileme olasılığı
p_renew = sigmoid(0.2 + 0.7 * z + 0.4 * voice + 0.5 * export)

# Hafta hafta: paid[:, k] = k+1. haftayı ödedi mi
paid = np.zeros((n, W), dtype=bool)
paid[:, 0] = start_week == 1
for k in range(1, W):
    weeks_so_far = paid[:, :k].sum(axis=1)
    # Heves kullanıcıları ilk yenilemeyi yapıyor, sonra toplu bırakıyor
    p = np.where(tourist, np.where(weeks_so_far == 1, 0.90, 0.20), p_renew)
    stays = rng.random(n) < p
    starts_now = start_week == k + 1
    paid[:, k] = (paid[:, k - 1] & stays) | starts_now

for k in range(W):
    users[f"paid_w{k+1}"] = paid[:, k]

# Gelir: ülkeye göre fiyat, net
price = users["country_tier"].map(countries["price_mult"]).to_numpy()
users["weeks_paid_w10"] = paid.sum(axis=1)
users["iap_revenue_w1"] = users["d0_iap"].to_numpy() * NET_IAP * price
users["value_w10"] = users["weeks_paid_w10"] * NET_WEEKLY * price + users["iap_revenue_w1"]

# Özet
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("Ödeme yapan kullanıcı:", int((users["weeks_paid_w10"] > 0).sum()), "| geç abone:", int(late.sum()))

d0s = users[users["d0_subscribed"]]
print("\nHaftalık retention, ilk gün aboneleri:")
print(d0s.groupby("channel")[[f"paid_w{k}" for k in range(1, W + 1)]].mean().round(2))

summary = pd.DataFrame({
    "spend": cells.groupby("channel")["spend"].sum(),
    "value_w10": users.groupby("channel")["value_w10"].sum(),
})
summary["roas_w10"] = summary["value_w10"] / summary["spend"]
print("\nTanrının gözü, kanal bazında 10. hafta:")
print(summary.round(2))

Ödeme yapan kullanıcı: 6703 | geç abone: 1131

Haftalık retention, ilk gün aboneleri:
          paid_w1  paid_w2  paid_w3  paid_w4  paid_w5  paid_w6  paid_w7  paid_w8  paid_w9  paid_w10
channel                                                                                            
Google        1.0     0.86     0.76     0.68     0.61     0.54     0.49     0.44     0.41      0.37
Meta          1.0     0.84     0.72     0.63     0.55     0.49     0.43     0.39     0.35      0.31
Snapchat      1.0     0.79     0.63     0.52     0.43     0.37     0.31     0.27     0.24      0.22
TikTok        1.0     0.87     0.38     0.25     0.20     0.16     0.15     0.13     0.12      0.10

Tanrının gözü, kanal bazında 10. hafta:
             spend  value_w10  roas_w10
channel                                
Google    40000.00   48646.02      1.22
Meta      40000.00   54030.27      1.35
Snapchat  28666.67   25695.48      0.90
TikTok    28666.67   26921.06      0.94


In [75]:
W = HORIZON_WEEKS
z = users["intent"].to_numpy()
voice = users["d0_voice_recorded"].to_numpy()
export = users["d0_song_exported"].to_numpy()
tourist = users["is_tourist"].to_numpy()
sub0 = users["d0_subscribed"].to_numpy()

# İlk gün abone olmayanların küçük bir kısmı ilk hafta içinde abone oluyor
late = ~sub0 & (rng.random(n) < sigmoid(-5.0 + 1.0 * z + 0.8 * export))
start_week = np.where(sub0, 1, np.where(late, 2, 0))   # 0 = hiç abone olmadı

# Her abonenin kişisel haftalık yenileme olasılığı
p_renew = sigmoid(0.2 + 0.7 * z + 0.4 * voice + 0.5 * export)

# Hafta hafta: paid[:, k] = k+1. haftayı ödedi mi
paid = np.zeros((n, W), dtype=bool)
paid[:, 0] = start_week == 1
for k in range(1, W):
    weeks_so_far = paid[:, :k].sum(axis=1)
    # Heves kullanıcıları ilk yenilemeyi yapıyor, sonra toplu bırakıyor
    p = np.where(tourist, np.where(weeks_so_far == 1, 0.90, 0.20), p_renew)
    stays = rng.random(n) < p
    starts_now = start_week == k + 1
    paid[:, k] = (paid[:, k - 1] & stays) | starts_now

for k in range(W):
    users[f"paid_w{k+1}"] = paid[:, k]

# Gelir: ülkeye göre fiyat, net
price = users["country_tier"].map(countries["price_mult"]).to_numpy()
users["weeks_paid_w10"] = paid.sum(axis=1)
users["iap_revenue_w1"] = users["d0_iap"].to_numpy() * NET_IAP * price
users["value_w10"] = users["weeks_paid_w10"] * NET_WEEKLY * price + users["iap_revenue_w1"]

# Özet
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("Ödeme yapan kullanıcı:", int((users["weeks_paid_w10"] > 0).sum()), "| geç abone:", int(late.sum()))

d0s = users[users["d0_subscribed"]]
print("\nHaftalık retention, ilk gün aboneleri:")
print(d0s.groupby("channel")[[f"paid_w{k}" for k in range(1, W + 1)]].mean().round(2))

summary = pd.DataFrame({
    "spend": cells.groupby("channel")["spend"].sum(),
    "value_w10": users.groupby("channel")["value_w10"].sum(),
})
summary["roas_w10"] = summary["value_w10"] / summary["spend"]
print("\nTanrının gözü, kanal bazında 10. hafta:")
print(summary.round(2))

Ödeme yapan kullanıcı: 6724 | geç abone: 1152

Haftalık retention, ilk gün aboneleri:
          paid_w1  paid_w2  paid_w3  paid_w4  paid_w5  paid_w6  paid_w7  paid_w8  paid_w9  paid_w10
channel                                                                                            
Google        1.0     0.89     0.80     0.71     0.63     0.56     0.50     0.47     0.42      0.38
Meta          1.0     0.85     0.73     0.63     0.55     0.48     0.43     0.38     0.34      0.32
Snapchat      1.0     0.78     0.63     0.52     0.44     0.36     0.33     0.28     0.24      0.22
TikTok        1.0     0.88     0.40     0.26     0.21     0.18     0.16     0.13     0.12      0.11

Tanrının gözü, kanal bazında 10. hafta:
             spend  value_w10  roas_w10
channel                                
Google    40000.00   50130.41      1.25
Meta      40000.00   54483.86      1.36
Snapchat  28666.67   25890.68      0.90
TikTok    28666.67   27388.07      0.96


In [76]:
# ---- Blok 9: case formatında hücre tablosu ve Simpson kontrolü ----
agg = users.groupby("cell_id").agg(
    installs=("user_id", "size"),
    subs=("d0_subscribed", "sum"),
    paid_w2=("paid_w2", "sum"),
    value_w10=("value_w10", "sum"),
)
cell_table = cells.join(agg[["subs", "paid_w2", "value_w10"]])
cell_table["install_to_sub"] = cell_table["subs"] / cell_table["installs"]
cell_table["cpp"] = cell_table["spend"] / cell_table["subs"]
cell_table["roas_w10"] = cell_table["value_w10"] / cell_table["spend"]

def roas_by(df, keys):
    g = df.groupby(keys)[["spend", "value_w10"]].sum()
    g["roas_w10"] = g["value_w10"] / g["spend"]
    return g["roas_w10"].round(2)

meta = cell_table[cell_table["channel"] == "Meta"]
print("Meta, toplamda kreatif ROAS:")
print(roas_by(meta, "archetype"))
print("\nMeta, ülke ülke kreatif ROAS:")
print(roas_by(meta, ["country_tier", "archetype"]).unstack())

# ---- Blok 10: ilk 24 saatten 10. hafta değeri ----
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Model sadece kurulum anında ve ilk 24 saatte bilinenleri görüyor.
# Kanal ve kreatif YOK: iOS'ta uygulama kullanıcının hangi reklamdan geldiğini bilmiyor.
features = d0_cols + ["country_tier", "platform"]
X = users[features].copy()
for c in ["country_tier", "platform"]:
    X[c] = X[c].astype("category")
for c in d0_cols:
    if X[c].dtype == bool:
        X[c] = X[c].astype(int)
y = users["value_w10"]

idx_train, idx_test = train_test_split(users.index, test_size=0.3, random_state=42)

model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31,
                          min_child_samples=50, random_state=42, verbose=-1)
model.fit(X.loc[idx_train], y.loc[idx_train])
users["pred_w10"] = model.predict(X).clip(min=0)

# Baseline: herkese kendi kanalının ortalaması
chan_mean = users.loc[idx_train].groupby("channel")["value_w10"].mean()
users["baseline_w10"] = users["channel"].map(chan_mean)

test = users.loc[idx_test]
print("Kullanıcı seviyesinde ortalama hata (MAE), test verisi:")
print("  Baseline:", round(mean_absolute_error(test["value_w10"], test["baseline_w10"]), 3), "$")
print("  Model   :", round(mean_absolute_error(test["value_w10"], test["pred_w10"]), 3), "$")

# Asıl soru: model aboneleri doğru ayırıyor mu? İlk gün abonelerinde tahmin ve gerçek
subs_test = test[test["d0_subscribed"]]
print("\nİlk gün aboneleri, kanal bazında ortalama 10. hafta değeri ($):")
print(pd.DataFrame({
    "gercek": subs_test.groupby("channel")["value_w10"].mean(),
    "model": subs_test.groupby("channel")["pred_w10"].mean(),
    "baseline_abone_ayni": subs_test["value_w10"].mean(),
}).round(2))

imp = pd.Series(model.booster_.feature_importance("gain"), index=features)
print("\nÖzellik önemi (toplam katkının yüzdesi):")
print((imp / imp.sum() * 100).sort_values(ascending=False).round(1))

# ---- Blok 11: hibrit SKAN şeması (event mapping) ----
# Aşama (2 bit): kullanıcının ilk gün ne yaptığı, kesin bilgi
users["stage"] = np.select(
    [users["d0_subscribed"] & users["d0_voice_recorded"] & users["d0_song_exported"],
     users["d0_subscribed"] & users["d0_iap"],
     users["d0_subscribed"]],
    [3, 2, 1], default=0)

# Kova (4 bit): modelin tahmini, her aşamanın içinde 16 eşit dilime bölünüyor
train = users.loc[idx_train]
edges = {s: np.unique(np.quantile(train.loc[train["stage"] == s, "pred_w10"],
                                  np.linspace(0, 1, 17)[1:-1]))
         for s in range(4)}
users["bucket"] = [int(np.searchsorted(edges[s], p, side="right"))
                   for s, p in zip(users["stage"], users["pred_w10"])]

users["cv"] = users["stage"] * 16 + users["bucket"]
users["coarse"] = np.where(users["stage"] == 0, "low",
                           np.where(users["bucket"] >= 8, "high", "medium"))

# Çözme tablosu: her sayının dolar karşılığı, eğitim verisinden
train = users.loc[idx_train]
decode_cv = train.groupby("cv")["value_w10"].mean()
decode_coarse = train.groupby("coarse")["value_w10"].mean()

schema = pd.DataFrame({"cv": range(64)})
schema["stage"] = schema["cv"] // 16
schema["bucket"] = schema["cv"] % 16
schema["stage_meaning"] = schema["stage"].map({
    0: "abone olmadı", 1: "abone oldu", 2: "abone ve IAP", 3: "abone, ses kaydı ve dışa aktarma"})
schema["coarse"] = np.where(schema["stage"] == 0, "low",
                            np.where(schema["bucket"] >= 8, "high", "medium"))
stage_mean = train.groupby("stage")["value_w10"].mean()
# Eğitimde hiç görülmeyen sayı varsa aşamanın ortalamasını kullan
schema["decoded_value_usd"] = schema["cv"].map(decode_cv).fillna(schema["stage"].map(stage_mean)).round(2)
decode_cv = schema.set_index("cv")["decoded_value_usd"]
print("Kullanılan farklı değer sayısı:", users["cv"].nunique(), "/ 64")
print(schema.iloc[[0, 8, 15, 16, 24, 31, 48, 56, 63]].to_string(index=False))
print("\nKaba değerlerin dolar karşılığı:", decode_coarse.round(2).to_dict())

Meta, toplamda kreatif ROAS:
archetype
demo    1.40
hype    1.21
ugc     1.47
Name: roas_w10, dtype: float64

Meta, ülke ülke kreatif ROAS:
archetype     demo  hype   ugc
country_tier                  
ABD           2.07  1.33  1.61
Tier1         1.93  1.09  1.50
Tier2         1.14  0.64  1.08
Kullanıcı seviyesinde ortalama hata (MAE), test verisi:
  Baseline: 2.489 $
  Model   : 0.934 $

İlk gün aboneleri, kanal bazında ortalama 10. hafta değeri ($):
          gercek  model  baseline_abone_ayni
channel                                     
Google     30.98  27.48                 24.9
Meta       26.14  24.70                 24.9
Snapchat   23.27  23.09                 24.9
TikTok     17.89  22.48                 24.9

Özellik önemi (toplam katkının yüzdesi):
d0_subscribed           78.1
country_tier             6.7
d0_song_exported         5.9
d0_mins_to_first_gen     3.6
d0_voice_recorded        2.2
d0_songs_generated       2.0
d0_iap                   0.6
platform                 0.4


In [77]:
# ---- Blok 12: SKAN simülatörü, karar vericinin gördüğü dünya ----
# Varsayım: iOS'ta bir hücre (kampanya) 250 install altındaysa Apple değer göndermiyor,
# 500 altındaysa sadece low/medium/high gönderiyor, üstündeyse 0-63 arası sayıyı gönderiyor.
NULL_BELOW, COARSE_BELOW = 250, 500

ios = users["platform"] == "iOS"
ios_cell_size = users[ios].groupby("cell_id").size()
tier = pd.cut(ios_cell_size, [0, NULL_BELOW, COARSE_BELOW, np.inf],
              labels=["yok", "kaba", "detaylı"], right=False)
users["skan_tier"] = np.where(ios, users["cell_id"].map(tier).astype(str), "android")
print("iOS hücreleri, Apple'ın gönderdiği bilgi:")
print(tier.value_counts().to_string())

price_u = users["country_tier"].map(countries["price_mult"])
spend_c = cells.groupby("channel")["spend"].sum()

# KÖR karar verici: Android'de ilk haftanın gerçek gelirini görür.
# iOS'ta SKAN sayısını sadece "abone mi değil mi" diye okur, boşluğu sıfır sayar.
# Sonra herkese aynı kuralı uygular: "abone ortalama 4 hafta öder".
RULE_OF_THUMB_WEEKS = 4
first_week_android = users["paid_w1"] * NET_WEEKLY * price_u + users["iap_revenue_w1"]
looks_subscribed = (users["skan_tier"].isin(["detaylı", "kaba"])
                    & np.where(users["skan_tier"] == "detaylı", users["stage"] >= 1, users["coarse"] != "low"))
first_week_ios = looks_subscribed * NET_WEEKLY * price_u
users["seen_blind"] = np.where(~ios, first_week_android, first_week_ios) * RULE_OF_THUMB_WEEKS

# TAHMİNE DAYALI karar verici: Android'de model tahmini, iOS'ta çözme tablosu.
# Değer gelmeyen hücrede kanalın görünen iOS hücrelerinin ortalamasıyla doldurur.
seen = np.select(
    [~ios, users["skan_tier"] == "detaylı", users["skan_tier"] == "kaba"],
    [users["pred_w10"], users["cv"].map(decode_cv), users["coarse"].map(decode_coarse)],
    default=np.nan)
users["seen_pred"] = seen
ios_visible_avg = users[ios & users["skan_tier"].ne("yok")].groupby("channel")["seen_pred"].mean()
missing = ios & (users["skan_tier"] == "yok")
users.loc[missing, "seen_pred"] = users.loc[missing, "channel"].map(ios_visible_avg)

est = pd.DataFrame({
    "kor": users.groupby("channel")["seen_blind"].sum() / spend_c,
    "tahmin": users.groupby("channel")["seen_pred"].sum() / spend_c,
    "tanri": users.groupby("channel")["value_w10"].sum() / spend_c,
})
print("\nHer karar vericinin gözünden 10. hafta ROAS:")
print(est.round(2))

# ---- Blok 13: bütçe dağıtımı ve hakem ----
BUDGET = 150_000
STEP = 1_000
ALPHA = 0.7   # azalan getiri: bütçe 2 katına çıkınca gelir 2^0.7 = 1.62 katına çıkar

def channel_value(roas_at_test, spend):
    s0 = spend_c
    return roas_at_test * s0 * (spend / s0) ** ALPHA

def allocate(roas_at_test):
    # Her 1.000 doları o an en yüksek marjinal getiriyi veren kanala ver
    alloc = pd.Series(0.0, index=roas_at_test.index)
    for _ in range(BUDGET // STEP):
        gain = channel_value(roas_at_test, alloc + STEP) - channel_value(roas_at_test, alloc)
        alloc[gain.idxmax()] += STEP
    return alloc

def referee(alloc):
    # Hakem her dağıtımı GERÇEK dünyada puanlar
    return (channel_value(est["tanri"], alloc).sum() - alloc.sum())

allocs = pd.DataFrame({k: allocate(est[k]) for k in ["kor", "tahmin", "tanri"]})
profits = pd.Series({k: referee(allocs[k]) for k in allocs.columns})

print("\nBütçe dağıtımı ($):")
print(allocs.astype(int))
print("\nGerçekleşen 10. hafta kârı ($):")
print(profits.round(0))
gap = (profits["tahmin"] - profits["kor"]) / (profits["tanri"] - profits["kor"])
print(f"\nKör yöntemle mükemmel bilgi arasındaki farkın %{gap*100:.0f}'i kapandı.")

def label(r):
    if r >= 1.2: return "Ölçekle"
    if r >= 1.0: return "Tut"
    return "Kapat ya da yenile"
print("\nEtiketler:", est["tahmin"].apply(label).to_dict())

# ---- Blok 14: duyarlılık analizi ----
# Sonuç azalan getiri varsayımına ve bütçe büyüklüğüne bağlı mı?
rows = []
for a in [0.6, 0.7, 0.8]:
    for b in [100_000, 150_000, 250_000]:
        ALPHA, BUDGET = a, b
        al = pd.DataFrame({k: allocate(est[k]) for k in ["kor", "tahmin", "tanri"]})
        p = pd.Series({k: referee(al[k]) for k in al.columns})
        rows.append({"alpha": a, "butce": b, "kor_kar": round(p["kor"]), "tahmin_kar": round(p["tahmin"]),
                     "tanri_kar": round(p["tanri"]),
                     "kapanan_fark_%": round((p["tahmin"] - p["kor"]) / (p["tanri"] - p["kor"]) * 100)})
ALPHA, BUDGET = 0.7, 150_000
print("\nDuyarlılık analizi:")
print(pd.DataFrame(rows).to_string(index=False))

iOS hücreleri, Apple'ın gönderdiği bilgi:
detaylı    26
kaba        8
yok         2

Her karar vericinin gözünden 10. hafta ROAS:
           kor  tahmin  tanri
channel                      
Google    0.71    1.08   1.25
Meta      0.85    1.26   1.36
Snapchat  0.62    0.97   0.90
TikTok    0.94    1.18   0.96

Bütçe dağıtımı ($):
            kor  tahmin  tanri
channel                       
Google    30000   36000  53000
Meta      53000   61000  69000
Snapchat  13000   18000  13000
TikTok    54000   35000  15000

Gerçekleşen 10. hafta kârı ($):
kor       14883.0
tahmin    19962.0
tanri     23138.0
dtype: float64

Kör yöntemle mükemmel bilgi arasındaki farkın %62'i kapandı.

Etiketler: {'Google': 'Tut', 'Meta': 'Ölçekle', 'Snapchat': 'Kapat ya da yenile', 'TikTok': 'Tut'}

Duyarlılık analizi:
 alpha  butce  kor_kar  tahmin_kar  tanri_kar  kapanan_fark_%
   0.6 100000    29139       31484      33190              58
   0.6 150000    14676       17816      19877              60
   0.6 25000

In [78]:
# ---- Blok 15: sonuçları kaydet ve indir ----
import os, shutil
os.makedirs("outputs", exist_ok=True)

schema.to_csv("outputs/skan_schema.csv", index=False)               # 64 satırlık şema ve çözme tablosu
cell_table.to_csv("outputs/cell_table.csv")                          # 72 hücre, case formatında
est.round(3).to_csv("outputs/roas_by_decision_maker.csv")            # üç gözden ROAS
allocs.to_csv("outputs/budget_allocation.csv")                       # üç dağıtım
profits.round(0).to_csv("outputs/profit_by_decision_maker.csv")      # hakemin puanları
pd.DataFrame(rows).to_csv("outputs/sensitivity.csv", index=False)    # duyarlılık analizi

shutil.make_archive("outputs", "zip", "outputs")
from google.colab import files
files.download("outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>